# 淘宝用户行为分析｜Jupyter 复现

目标：用 Pandas 重新做一遍核心分析，再用 Matplotlib / Seaborn 做可视化。

> 数据源为本地原始 CSV（`../data/raw/淘宝.csv`）；代码按时间戳筛选窗口，并未显式去重，因此不保证与 SQL `taobao_clean` 完全一致。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

# 中文显示按自己电脑字体调整
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False


## 1. 读取数据（本地 CSV，替代 MySQL）

原 notebook 从 MySQL 的 `taobao_clean` 表读数据；这里改为直读本地原始 CSV，
并用与 `01_taobao_business_analysis.sql` 一致的时间窗口过滤。


In [ ]:
import datetime

csv_path = r"../data/raw/淘宝.csv"

# 时区：与 SQL 的 SET time_zone='+08:00' 一致，按北京时间(UTC+8)解析
CST = datetime.timezone(datetime.timedelta(hours=8))

# 与 SQL 一致的时间窗口：2017-11-24 00:00:00 ~ 2017-11-30 23:59:59（北京时间）
lo = int(datetime.datetime(2017, 11, 24, 0, 0, 0, tzinfo=CST).timestamp())
hi = int(datetime.datetime(2017, 11, 30, 23, 59, 59, tzinfo=CST).timestamp())

cols = ["user_id", "item_id", "category_id", "behavior", "timestamp"]
dtypes = {
    "user_id": "int32",
    "item_id": "int64",
    "category_id": "int32",
    "behavior": "category",
    "timestamp": "int64",
}

# 分块读取 + 时间窗口过滤，降低内存占用
chunks = []
for chunk in pd.read_csv(csv_path, header=None, names=cols, dtype=dtypes, chunksize=5_000_000):
    chunk = chunk[(chunk["timestamp"] >= lo) & (chunk["timestamp"] <= hi)]
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
del chunks

# 派生时间字段（对应 taobao_clean 的 dt / day_date / hour_num）
df["dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
df["day_date"] = df["dt"].dt.normalize()
df["hour_num"] = df["dt"].dt.hour
df = df.drop(columns=["timestamp"])

df.head()


In [ ]:
# 基础检查
print(df.shape)
print(df.dtypes)
print(df["behavior"].value_counts())
print(df.isna().sum())


## 2. 业务问题一：用户卡在哪一步？

先看有多少用户发生过浏览、收藏、加购、购买。


In [ ]:
user_behavior = (
    df.assign(flag=1)
      .pivot_table(
          index="user_id",
          columns="behavior",
          values="flag",
          aggfunc="max",
          fill_value=0
      )
      .reset_index()
)

for c in ["pv", "fav", "cart", "buy"]:
    if c not in user_behavior.columns:
        user_behavior[c] = 0

funnel = pd.DataFrame({
    "stage": ["浏览", "收藏", "加购", "购买"],
    "users": [
        user_behavior["pv"].sum(),
        user_behavior["fav"].sum(),
        user_behavior["cart"].sum(),
        user_behavior["buy"].sum()
    ]
})

funnel


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=funnel, x="stage", y="users")
plt.title("用户行为覆盖")
plt.xlabel("")
plt.ylabel("用户数")
plt.tight_layout()
plt.show()


In [ ]:
# 看加购但没购买的人有多少
cart_not_buy_user = user_behavior[
    (user_behavior["cart"] == 1) &
    (user_behavior["buy"] == 0)
]

print("加购未购买用户数：", len(cart_not_buy_user))


### 我怎么解释

- 如果浏览用户很多，但加购用户明显少，先看商品吸引力、详情页或推荐质量。
- 如果加购用户多，但购买明显少，购物车召回更值得继续分析。
- 这里只能定位现象，不能直接说是哪一个原因导致。


## 3. 业务问题二：流量高峰是不是购买高峰？


In [ ]:
hourly = (
    df.groupby("hour_num")
      .agg(
          pv_cnt=("behavior", lambda x: (x == "pv").sum()),
          buy_cnt=("behavior", lambda x: (x == "buy").sum()),
          active_user=("user_id", "nunique")
      )
      .reset_index()
)

pv_users = (
    df[df["behavior"] == "pv"]
    .groupby("hour_num")["user_id"]
    .nunique()
    .rename("pv_user")
)

buy_users = (
    df[df["behavior"] == "buy"]
    .groupby("hour_num")["user_id"]
    .nunique()
    .rename("buy_user")
)

hourly = hourly.merge(pv_users, on="hour_num", how="left")
hourly = hourly.merge(buy_users, on="hour_num", how="left").fillna(0)
hourly["viewer_to_buyer_rate"] = hourly["buy_user"] / hourly["pv_user"].replace(0, pd.NA)

hourly.head()


In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly, x="hour_num", y="pv_cnt", marker="o", label="PV")
sns.lineplot(data=hourly, x="hour_num", y="buy_cnt", marker="o", label="购买")
plt.title("24小时浏览与购买趋势")
plt.xlabel("小时")
plt.ylabel("行为次数")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=hourly, x="hour_num", y="viewer_to_buyer_rate", marker="o")
plt.title("24小时浏览用户到购买用户比例")
plt.xlabel("小时")
plt.ylabel("比例")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()


### 我怎么解释

重点不是找“最高峰”，而是找：

- PV 很高，但购买用户比例偏低的时段；
- 流量不算高，但购买表现比较好的时段。

前者适合继续排查流量质量，后者可以考虑是否有增加曝光的空间。


## 4. 业务问题三：哪些类目值得优先看？


In [ ]:
category_event = (
    df.groupby("category_id")
      .agg(
          pv_cnt=("behavior", lambda x: (x == "pv").sum()),
          cart_cnt=("behavior", lambda x: (x == "cart").sum()),
          buy_cnt=("behavior", lambda x: (x == "buy").sum())
      )
      .reset_index()
)

# 每个类目：有浏览的用户数 / 有购买的用户数 / 既浏览又购买的用户数
# （原 notebook 用 pivot_table 会物化 category×user 笛卡尔积导致内存溢出，
#   这里改用等价的分组去重计数，结果与 SQL 的 SUM(has_pv)/SUM(has_buy) 一致）
pv_user = (
    df[df["behavior"] == "pv"]
    .groupby("category_id")["user_id"]
    .nunique()
    .rename("pv_user")
)
buy_user = (
    df[df["behavior"] == "buy"]
    .groupby("category_id")["user_id"]
    .nunique()
    .rename("buy_user")
)

# 同一 (类目, 用户) 内出现 pv 和 buy 两种行为 → 既浏览又购买
pv_and_buy_user = (
    df[df["behavior"].isin(["pv", "buy"])]
    .groupby(["category_id", "user_id"])["behavior"]
    .nunique()
)
pv_and_buy_user = (
    pv_and_buy_user[pv_and_buy_user >= 2]
    .groupby("category_id")
    .size()
    .rename("pv_and_buy_user")
)

category_user = pd.DataFrame({
    "pv_user": pv_user,
    "buy_user": buy_user,
    "pv_and_buy_user": pv_and_buy_user,
}).fillna(0).astype(int).reset_index()

category = category_event.merge(category_user, on="category_id", how="left")
category["viewer_to_buyer_rate"] = (
    category["pv_and_buy_user"] / category["pv_user"].replace(0, pd.NA)
)

category.sort_values("pv_cnt", ascending=False).head(10)


In [ ]:
# 只画流量较高的类目，避免点太密
plot_cat = category.nlargest(50, "pv_cnt").copy()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_cat,
    x="pv_cnt",
    y="viewer_to_buyer_rate",
    size="buy_cnt",
    sizes=(20, 300),
    alpha=0.7
)
plt.title("类目流量与购买表现")
plt.xlabel("PV")
plt.ylabel("浏览用户到购买用户比例")
plt.tight_layout()
plt.show()


In [ ]:
# 找高流量低转化类目
pv_q75 = category["pv_cnt"].quantile(0.75)
rate_median = category["viewer_to_buyer_rate"].median()

problem_category = category[
    (category["pv_cnt"] >= pv_q75) &
    (category["viewer_to_buyer_rate"] <= rate_median)
].sort_values("pv_cnt", ascending=False)

problem_category.head(15)


### 我怎么解释

高流量低转化类目优先级更高，因为已经有流量基础。

但是当前数据没有价格、活动、库存、详情页曝光等字段，所以这里只能定位“哪些类目值得继续查”，不能直接判断原因。


## 5. 业务问题四：购物车流失集中在哪些类目？


In [ ]:
cart = (
    df[df["behavior"] == "cart"]
    .groupby(["user_id", "item_id", "category_id"])
    .agg(
        first_cart_time=("dt", "min"),
        cart_cnt=("behavior", "size")
    )
    .reset_index()
)

buy = (
    df[df["behavior"] == "buy"]
    .groupby(["user_id", "item_id"])
    .agg(first_buy_time=("dt", "min"))
    .reset_index()
)

cart_buy = cart.merge(buy, on=["user_id", "item_id"], how="left")

lost = cart_buy[
    cart_buy["first_buy_time"].isna() |
    (cart_buy["first_buy_time"] < cart_buy["first_cart_time"])
].copy()

lost_category = (
    lost.groupby("category_id")
    .agg(
        lost_user_item=("item_id", "size"),
        lost_user=("user_id", "nunique"),
        lost_item=("item_id", "nunique")
    )
    .reset_index()
    .sort_values("lost_user_item", ascending=False)
)

lost_category.head(10)


In [ ]:
top_lost = lost_category.head(10)

plt.figure(figsize=(9, 5))
sns.barplot(data=top_lost, y="category_id", x="lost_user_item", orient="h")
plt.title("加购未购买最多的 Top10 类目")
plt.xlabel("用户-商品组合数")
plt.ylabel("类目")
plt.tight_layout()
plt.show()


## 6. 业务问题五：哪些用户值得重点运营？


In [ ]:
user_value = (
    df.groupby("user_id")
      .agg(
          active_days=("day_date", "nunique"),
          pv_cnt=("behavior", lambda x: (x == "pv").sum()),
          cart_cnt=("behavior", lambda x: (x == "cart").sum()),
          buy_cnt=("behavior", lambda x: (x == "buy").sum())
      )
      .reset_index()
)

q_active = user_value["active_days"].quantile(0.75)
q_buy = user_value["buy_cnt"].quantile(0.75)

def segment(row):
    if row["cart_cnt"] > 0 and row["buy_cnt"] == 0:
        return "加购未购买"
    if row["active_days"] >= q_active and row["buy_cnt"] >= q_buy:
        return "高活跃高购买"
    if row["active_days"] >= q_active and row["buy_cnt"] < q_buy:
        return "高活跃低购买"
    if row["active_days"] < q_active and row["buy_cnt"] >= q_buy:
        return "高购买低活跃"
    return "普通用户"

user_value["user_type"] = user_value.apply(segment, axis=1)
user_value["user_type"].value_counts()


In [ ]:
segment_count = (
    user_value["user_type"]
    .value_counts()
    .rename_axis("user_type")
    .reset_index(name="users")
)

plt.figure(figsize=(9, 4))
sns.barplot(data=segment_count, x="users", y="user_type")
plt.title("用户分层")
plt.xlabel("用户数")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 7. 每日趋势

看每天流量和购买有没有一起变化。


In [ ]:
daily = (
    df.groupby("day_date")
      .agg(
          pv_cnt=("behavior", lambda x: (x == "pv").sum()),
          active_user=("user_id", "nunique"),
          buy_cnt=("behavior", lambda x: (x == "buy").sum())
      )
      .reset_index()
)

daily_buy_user = (
    df[df["behavior"] == "buy"]
    .groupby("day_date")["user_id"]
    .nunique()
    .rename("buy_user")
    .reset_index()
)

daily = daily.merge(daily_buy_user, on="day_date", how="left").fillna({"buy_user": 0})
daily["active_to_buy_rate"] = daily["buy_user"] / daily["active_user"]

daily


In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=daily, x="day_date", y="pv_cnt", marker="o")
plt.title("每日PV趋势")
plt.xlabel("日期")
plt.ylabel("PV")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=daily, x="day_date", y="active_to_buy_rate", marker="o")
plt.title("每日购买参与率")
plt.xlabel("日期")
plt.ylabel("购买用户 / 活跃用户")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 8. 最后写结论时只回答 5 件事

1. 最大的转化损失出现在哪里？
2. 哪些小时高流量但购买弱？
3. 哪些类目高流量低转化，值得优先排查？
4. 哪些类目的加购未购买更集中？
5. 哪些用户适合做召回或促购？

不要把“相关”直接写成“原因”。
